<a href="https://colab.research.google.com/github/joansantoso/jax-ml/blob/main/DevFest26JKT_Create_Dataset_NovaMart_ID.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📦 Notebook 1 · Build the NovaMart customer-service dataset (Bahasa Indonesia)
By Joan Santoso (AI and Cloud AI GDE)

This notebook generates a **synthetic, tool-calling, reasoning** dataset for **Ava**, the customer-service agent of the fictional store *NovaMart*, and saves it to **Google Drive as JSONL**. **Notebook 2** loads it to fine-tune Gemma 4 on a TPU and serve it with vLLM.

**Runtime:** any (CPU is enough). No GPU/TPU, no model download, no Hugging Face token.

### What you get in Drive
```
MyDrive/gemma4-support-agent/data/
├── train.jsonl     # one conversation per line (90%)
├── test.jsonl      # held-out conversations (10%) — split by conversation, so no leakage
└── metadata.json   # system prompt, tools, reasoning tag, counts, seed, schema
```

### One record (one line of `train.jsonl`)
```json
{"id": "conv-00042", "intent": "refund", "split": "train",
 "messages": [
   {"role": "user", "content": "Saya mau mengembalikan air fryer dari pesanan NM-258582, saya berubah pikiran."},
   {"role": "assistant",
    "reasoning": "Pelanggan ingin retur pesanan NM-258582 ... Buka refund dengan start_refund ...",
    "content": "<tool_call>{\"name\": \"start_refund\", \"arguments\": {\"order_id\": \"NM-258582\", \"reason\": \"changed_mind\"}}</tool_call>"},
   {"role": "user", "content": "TOOL_RESULT: {\"refund_id\": \"RF-41234\", \"status\": \"approved\", ...}"},
   {"role": "assistant", "reasoning": "Refund RF-41234 disetujui ...", "content": "Beres! Saya sudah membuka retur ..."}
 ]}
```
* **Agentic:** the assistant calls a tool by replying *only* with `<tool_call>{...}</tool_call>`; the app answers with a `TOOL_RESULT:` message.
* **Reasoning:** every assistant turn has a short Indonesian `reasoning` — *why* this step — kept in its own field, so Notebook 2 can train with or without it.

## 1. Settings
Change `N_CONVERSATIONS` or `SEED` for a different dataset; the same seed always produces identical files. Set `SAVE_TO_DRIVE = False` to write to `/content/data` instead.

In [ ]:
import os
N_CONVERSATIONS = 1500
SEED            = 7
TEST_FRACTION   = 0.10
DATASET_NAME    = "novamart-cs-id"
DATASET_VERSION = "2.0"
SAVE_TO_DRIVE   = True
DRIVE_DATA_DIR  = "/content/drive/MyDrive/gemma4-support-agent/data"
LOCAL_DATA_DIR  = "/content/data"

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
OUT_DIR = DRIVE_DATA_DIR if SAVE_TO_DRIVE else LOCAL_DATA_DIR
os.makedirs(OUT_DIR, exist_ok=True)
print("Writing the dataset to", OUT_DIR)

Mounted at /content/drive
Writing the dataset to /content/drive/MyDrive/gemma4-support-agent/data


## 2. The generator
Nine intents, each a Python function that returns one conversation:

| Intent | What happens | Tools |
|---|---|---|
| `order_status` | Order id given → lookup → status answer (12% unknown ids) | `lookup_order` |
| `order_no_id` | No id → Ava asks for it → lookup → answer | `lookup_order` |
| `refund` | Return with a reason → refund (20% outside the 30-day window) | `start_refund` |
| `damaged` | Broken item → Ava asks refund or replacement → refund, or escalate for replacement | `start_refund` / `escalate` |
| `password` | Login trouble → ask email if missing → reset link | `reset_password` |
| `escalate` | Customer (sometimes angry) wants a human | `escalate` |
| `faq` | Returns, shipping, payments, refunds, free shipping — answered from policy | — |
| `out_of_scope` | Essays, weather, elections, recipes… → polite refusal | — |
| `smalltalk` | Greetings, “are you a bot?” | — |

25% of conversations get a closing “makasih!” exchange. Customer messages are roughened (casing, punctuation, greetings, slang like *kak*, *min*, *dong*); names, products, ids, couriers, dates and Rupiah amounts are random. Every assistant turn is created with `A(content, reasoning)`.

In [ ]:
import random, json, re
from datetime import date, timedelta
random.seed(SEED)

# ============================================================
#  NovaMart customer-service conversations (Bahasa Indonesia)
#  Every assistant turn has:
#    content   – what Ava says or does (a reply, or ONLY a <tool_call>)
#    reasoning – a short Indonesian rationale for that step (the "think" part)
#  Tool names, JSON keys and the TOOL_RESULT prefix stay English: code parses them.
# ============================================================
COMPANY = "NovaMart"
SYSTEM_PROMPT = f"""Kamu adalah Ava, agen layanan pelanggan {COMPANY}, toko online elektronik dan perlengkapan rumah.
Bersikaplah ramah, singkat, dan akurat. Jangan pernah mengarang detail pesanan: gunakan tool.

Tool:
- lookup_order(order_id): cek status pesanan
- start_refund(order_id, reason): buka pengembalian/refund. reason salah satu dari: defective, wrong_item, changed_mind, damaged
- reset_password(email): kirim tautan reset kata sandi
- escalate(summary): alihkan percakapan ke agen manusia

Untuk memanggil tool, balas HANYA dengan: <tool_call>{{"name": "...", "arguments": {{...}}}}</tool_call>
Hasil tool dikirim dalam pesan yang diawali TOOL_RESULT:.

Kebijakan: pengembalian maksimal 30 hari setelah barang diterima; dana refund kembali ke metode pembayaran awal dalam 5-7 hari kerja;
pengiriman reguler 3-5 hari kerja (gratis ongkir untuk belanja di atas Rp500.000), ekspres 1-2 hari kerja;
pembayaran: transfer bank, kartu kredit, GoPay, OVO, dan QRIS.
Tolak dengan sopan permintaan yang tidak berkaitan dengan {COMPANY}. Selalu jawab dalam Bahasa Indonesia."""

REASONING_TAG = "reasoning"
REASONING_INSTRUCTION = ("Sebelum setiap balasan, tulis alasan singkat di dalam <reasoning>...</reasoning>, "
                         "lalu tulis tool call atau jawaban untuk pelanggan.")

TOOLS = [
    {"name": "lookup_order", "description": "Cek status pesanan",
     "parameters": {"order_id": "string, format NM-123456"}},
    {"name": "start_refund", "description": "Buka pengembalian/refund",
     "parameters": {"order_id": "string", "reason": "defective | wrong_item | changed_mind | damaged"}},
    {"name": "reset_password", "description": "Kirim tautan reset kata sandi",
     "parameters": {"email": "string"}},
    {"name": "escalate", "description": "Alihkan ke agen manusia",
     "parameters": {"summary": "string, ringkasan singkat dalam Bahasa Indonesia"}},
]

FIRST_NAMES = ["Budi","Siti","Andi","Dewi","Rizky","Putri","Agus","Ayu","Fajar","Nurul","Dimas","Rina","Bayu","Intan","Yoga","Maya"]
PRODUCTS = ["earbud nirkabel","monitor 4K","air fryer","keyboard mekanik","robot penyedot debu","jam tangan pintar",
            "mesin espresso","mouse gaming","meja berdiri","speaker Bluetooth","hub USB-C","ketel listrik"]
CARRIERS = ["JNE","J&T Express","SiCepat","AnterAja"]
BULAN = ["Jan","Feb","Mar","Apr","Mei","Jun","Jul","Agu","Sep","Okt","Nov","Des"]
TODAY = date(2026, 11, 1)

def cap(s): return s[0].upper() + s[1:]
def fmt_tgl(d): return f"{d.day} {BULAN[d.month - 1]}"
def a_date(lo=-6, hi=8): return fmt_tgl(TODAY + timedelta(days=random.randint(lo, hi)))
def rupiah(x): return "Rp" + f"{x:,}".replace(",", ".")
def a_price(): return random.randrange(150_000, 8_000_000, 1_000)
def order_id():  return f"NM-{random.randint(100000, 999999)}"
def email_for(n): return f"{n.lower()}{random.randint(1,99)}@{random.choice(['gmail.com','yahoo.co.id','outlook.com','proton.me'])}"
def tool_call(name, **args): return "<tool_call>" + json.dumps({"name": name, "arguments": args}, ensure_ascii=False) + "</tool_call>"
def tool_result(obj): return "TOOL_RESULT: " + json.dumps(obj, ensure_ascii=False)
def U(t): return {"role": "user", "content": t}
def A(t, why): return {"role": "assistant", "content": t, "reasoning": why}

def noisy(t):
    """Light augmentation: casing, punctuation, greetings and chat slang like 'kak' / 'min'."""
    r = random.random()
    if r < 0.15:   t = t.lower()
    elif r < 0.25: t = t.rstrip("?.!")
    if random.random() < 0.2:
        t = random.choice(["Halo, ", "Hai ", "Permisi, ", "Selamat pagi. ", "Kak, "]) + t[0].lower() + t[1:]
    if random.random() < 0.15:
        t = t.rstrip("?.!") + random.choice([" kak", " min", " ya", " dong"])
    return t

# ---------- shared pieces ----------
def lookup_why(oid):
    return random.choice([f"Pelanggan menanyakan pesanan {oid}. Saya tidak boleh menebak statusnya, jadi saya cek dengan lookup_order.",
                          f"Ada nomor pesanan {oid}. Status harus dicek lewat lookup_order sebelum saya menjawab."])

def status_answer(oid, item):
    status = random.choice(["processing", "shipped", "out_for_delivery", "delivered"])
    res = {"order_id": oid, "item": item, "status": status}
    if status == "processing":
        res["ships_by"] = a_date(1, 4)
        ans = random.choice([
            f"Pesanan {oid} ({item}) sedang disiapkan dan dijadwalkan dikirim paling lambat {res['ships_by']}. Nomor resi akan kami kirim lewat email begitu paket berangkat dari gudang ya, Kak.",
            f"{cap(item)} Kakak (pesanan {oid}) masih dalam proses dan akan dikirim paling lambat {res['ships_by']}. Nomor resi akan dikirim lewat email setelah paket dikirim."])
        why = f"Status processing, dikirim paling lambat {res['ships_by']}. Sampaikan jadwalnya dan bahwa resi menyusul lewat email."
    elif status == "shipped":
        res.update(carrier=random.choice(CARRIERS), tracking=f"RESI{random.randint(10**8, 10**9-1)}", eta=a_date(1, 5))
        ans = random.choice([
            f"Kabar baik! {cap(item)} Kakak sudah dikirim via {res['carrier']} (resi {res['tracking']}) dan diperkirakan tiba pada {res['eta']}.",
            f"Pesanan {oid} sedang dalam perjalanan! {res['carrier']} memperkirakan {item} Kakak tiba pada {res['eta']}. Nomor resi: {res['tracking']}."])
        why = f"Paket sudah dikirim via {res['carrier']}, estimasi tiba {res['eta']}. Sampaikan kurir, nomor resi, dan perkiraan tiba."
    elif status == "out_for_delivery":
        res["carrier"] = random.choice(CARRIERS)
        ans = f"Kabar baik! {cap(item)} Kakak sedang diantar kurir {res['carrier']} hari ini. Mohon pastikan ada yang menerima paketnya ya."
        why = f"Paket sedang diantar {res['carrier']} hari ini. Beri tahu pelanggan agar ada yang menerima."
    else:
        res["delivered_on"] = a_date(-6, 0)
        ans = (f"Menurut data kami, {item} Kakak (pesanan {oid}) sudah diterima pada {res['delivered_on']}. "
               "Jika paketnya belum ditemukan, coba cek ke tetangga atau satpam/resepsionis gedung, lalu kabari saya kalau masih hilang. Saya bisa minta rekan kami untuk menyelidikinya.")
        why = f"Data menunjukkan paket diterima {res['delivered_on']}, tapi pelanggan mungkin belum menemukannya. Sarankan cek sekitar dan tawarkan penyelidikan."
    return res, ans, why

# ---------- intent generators ----------
def conv_order_status():
    oid, item = order_id(), random.choice(PRODUCTS)
    q = random.choice([f"Pesanan saya {oid} sudah sampai mana?", f"Bisa cek status pesanan {oid}?",
                       f"{cap(item)} saya belum datang. Nomor pesanannya {oid}.", f"cek resi pesanan {oid} dong",
                       f"Ada update untuk {oid}? Sudah lama nih."])
    msgs = [U(noisy(q)), A(tool_call("lookup_order", order_id=oid), lookup_why(oid))]
    if random.random() < 0.12:
        return msgs + [U(tool_result({"error": "order_not_found", "order_id": oid})),
                       A(f"Maaf Kak, saya tidak menemukan pesanan dengan nomor {oid}. Boleh dicek lagi nomornya? Nomor pesanan ada di email konfirmasi dan diawali NM-.",
                         f"Tool tidak menemukan pesanan {oid}. Kemungkinan nomornya salah ketik; minta pelanggan memeriksa ulang.")]
    res, ans, why = status_answer(oid, item)
    return msgs + [U(tool_result(res)), A(ans, why)]

def conv_order_no_id():
    oid, item = order_id(), random.choice(PRODUCTS)
    q = random.choice(["Paket saya sudah sampai mana ya?", f"{cap(item)} saya belum sampai juga.", "Bisa cek status pesanan saya?",
                       "Kapan barang saya datang??", f"Minggu lalu saya pesan {item}, ada kabar?"])
    ask = random.choice(["Dengan senang hati saya cek, Kak! Boleh minta nomor pesanannya? Nomornya diawali NM- dan ada di email konfirmasi.",
                         "Siap, Kak! Nomor pesanannya berapa? Formatnya seperti NM-123456 dan bisa ditemukan di email konfirmasi pesanan."])
    give = random.choice([f"Ini {oid}", oid, f"oke, nomor pesanannya {oid}", f"nih: {oid}"])
    res, ans, why = status_answer(oid, item)
    return [U(noisy(q)),
            A(ask, "Pelanggan ingin cek pesanan tapi belum memberi nomor pesanan. lookup_order butuh order_id, jadi saya minta nomornya dulu."),
            U(give), A(tool_call("lookup_order", order_id=oid), lookup_why(oid)), U(tool_result(res)), A(ans, why)]

REASONS = {"defective":    ["barangnya rusak setelah dua hari dipakai", "tidak bisa menyala", "koneksinya sering putus"],
           "wrong_item":   ["model yang dikirim salah", "saya menerima barang yang tidak saya pesan"],
           "changed_mind": ["saya berubah pikiran", "ternyata saya tidak membutuhkannya", "ukurannya tidak muat di ruangan saya"]}

def conv_refund():
    oid, item = order_id(), random.choice(PRODUCTS)
    reason = random.choice(list(REASONS)); why_txt = random.choice(REASONS[reason])
    q = random.choice([f"Saya mau mengembalikan {item} dari pesanan {oid}, {why_txt}.",
                       f"Bisa refund untuk pesanan {oid}? {cap(why_txt)}.",
                       f"Pengajuan retur {oid}: {why_txt}."])
    msgs = [U(noisy(q)), A(tool_call("start_refund", order_id=oid, reason=reason),
                           f"Pelanggan ingin retur pesanan {oid} karena \"{why_txt}\", yang masuk kategori {reason}. Buka refund dengan start_refund; sistem akan mengecek batas 30 hari.")]
    if random.random() < 0.2:
        delivered = a_date(-90, -40)
        res = {"error": "outside_return_window", "order_id": oid, "delivered_on": delivered}
        ans = (f"Mohon maaf, Kak. Pesanan {oid} diterima pada {delivered}, sudah melewati batas pengembalian 30 hari, jadi saya tidak bisa membuka retur biasa. "
               f"Jika {item} Kakak bermasalah, kemungkinan masih tercakup garansi pabrik. Mau saya hubungkan dengan rekan kami yang bisa membantu soal garansi?")
        why = f"Refund ditolak: barang diterima {delivered}, lebih dari 30 hari. Jelaskan kebijakannya dengan sopan dan tawarkan bantuan garansi."
    else:
        amount = a_price()
        res = {"refund_id": f"RF-{random.randint(10000, 99999)}", "order_id": oid, "status": "approved", "amount": amount, "currency": "IDR", "return_label": "emailed"}
        ans = random.choice([
            f"Beres! Saya sudah membuka retur {res['refund_id']} untuk {item} Kakak. Label retur prabayar sedang dikirim ke email Kakak. Setelah barang kami terima, dana {rupiah(amount)} akan dikembalikan ke metode pembayaran awal dalam 5-7 hari kerja.",
            f"Pengembalian Kakak sudah disetujui (nomor referensi {res['refund_id']}). Silakan gunakan label prabayar yang baru kami kirim ke email, dan dana {rupiah(amount)} akan dikembalikan ke metode pembayaran awal dalam 5-7 hari kerja setelah {item} kami terima."])
        why = f"Refund {res['refund_id']} disetujui sebesar {rupiah(amount)}. Sampaikan nomor referensi, label retur di email, dan waktu 5-7 hari kerja."
    return msgs + [U(tool_result(res)), A(ans, why)]

def conv_damaged():
    oid, item = order_id(), random.choice(PRODUCTS)
    q = random.choice([f"{cap(item)} saya datang dalam keadaan rusak. Pesanan {oid}.",
                       f"Kardusnya penyok dan {item}-nya retak :( pesanan {oid}",
                       f"Pesanan {oid}: {item}-nya pecah waktu sampai."])
    ask = random.choice([f"Mohon maaf sekali {item} Kakak tiba dalam keadaan rusak. Itu bukan pengalaman yang kami harapkan. Kakak lebih memilih penggantian barang atau refund penuh?",
                         f"Aduh, mohon maaf ya, Kak! Saya bisa langsung bantu: Kakak mau {item} pengganti atau refund?"])
    msgs = [U(noisy(q)), A(ask, "Barang tiba rusak. Ada dua jalan: refund (start_refund) atau penggantian (lewat tim manusia). Tanyakan dulu pilihan pelanggan.")]
    if random.random() < 0.5:
        amount, rid = a_price(), f"RF-{random.randint(10000, 99999)}"
        msgs += [U(random.choice(["Refund saja.", "refund", "Saya mau uangnya kembali saja."])),
                 A(tool_call("start_refund", order_id=oid, reason="damaged"),
                   f"Pelanggan memilih refund untuk barang rusak. Panggil start_refund untuk {oid} dengan reason damaged."),
                 U(tool_result({"refund_id": rid, "order_id": oid, "status": "approved", "amount": amount, "currency": "IDR", "return_label": "emailed"})),
                 A(f"Sudah beres: refund {rid} disetujui. Label prabayar untuk {item} yang rusak sudah kami kirim ke email, dan dana {rupiah(amount)} akan kembali ke metode pembayaran awal dalam 5-7 hari kerja.",
                   f"Refund {rid} disetujui. Sampaikan label prabayar dan dana {rupiah(amount)} dalam 5-7 hari kerja.")]
    else:
        tid = f"T-{random.randint(10000, 99999)}"
        msgs += [U(random.choice(["Ganti barang saja.", "penggantian", "Tolong kirim yang baru."])),
                 A(tool_call("escalate", summary=f"{cap(item)} rusak pada pesanan {oid}; pelanggan minta penggantian."),
                   "Pelanggan memilih penggantian. Tidak ada tool untuk mengirim barang pengganti, jadi saya eskalasi ke tim manusia dengan ringkasan."),
                 U(tool_result({"ticket_id": tid, "wait_minutes": random.randint(30, 240)})),
                 A(f"Saya sudah meneruskan ini ke tim penggantian (tiket {tid}). Mereka akan segera menghubungi Kakak lewat email untuk konfirmasi pengiriman {item} baru, tanpa biaya tambahan.",
                   f"Tiket {tid} dibuat untuk penggantian. Sampaikan nomor tiket dan bahwa tim akan menghubungi lewat email.")]
    return msgs

def conv_password():
    em = email_for(random.choice(FIRST_NAMES))
    if random.random() < 0.5:
        q = random.choice(["Saya tidak bisa masuk ke akun saya.", "lupa kata sandi", "Bagaimana cara reset password?", "Akun saya terkunci."])
        msgs = [U(noisy(q)), A("Tidak masalah, saya bantu ya, Kak. Alamat email akun Kakak apa?",
                               "Pelanggan perlu reset kata sandi. reset_password butuh email akun, jadi saya tanyakan dulu."),
                U(random.choice([em, f"emailnya {em}", f"{em} makasih"]))]
    else:
        msgs = [U(random.choice([f"Tolong reset password untuk {em}", f"Saya perlu reset kata sandi, email saya {em}"]))]
    return msgs + [A(tool_call("reset_password", email=em), f"Email akun sudah diberikan: {em}. Kirim tautan reset dengan reset_password."),
                   U(tool_result({"status": "sent", "email": em})),
                   A(f"Tautan reset kata sandi sudah saya kirim ke {em}. Tautannya berlaku 30 menit. Kalau tidak muncul di kotak masuk, coba cek folder spam ya, Kak.",
                     "Tautan reset terkirim. Sampaikan masa berlaku 30 menit dan sarankan cek folder spam.")]

ESCALATIONS = [("kesalahan tagihan di pesanan terakhir saya", "kesalahan tagihan pesanan terakhir"),
               ("tagihan yang terpotong dua kali", "tagihan ganda"),
               ("paket yang tidak pernah sampai", "paket tidak pernah sampai"),
               ("klaim garansi", "klaim garansi"),
               ("penutupan akun bisnis saya", "penutupan akun bisnis")]

def conv_escalate():
    user_topic, summary_topic = random.choice(ESCALATIONS)
    q = random.choice([f"Saya mau bicara dengan orang sungguhan soal {user_topic}.",
                       f"Keterlaluan. Sudah tiga kali saya menghubungi kalian soal {user_topic}. Sambungkan ke manajer!",
                       f"Bisa bicara dengan agen manusia? Ini soal {user_topic}."])
    tid, wait = f"T-{random.randint(10000, 99999)}", random.randint(2, 20)
    ans = random.choice([
        f"Saya sangat memahami, Kak, dan mohon maaf atas ketidaknyamanannya. Saya sudah meneruskan ini ke rekan kami. Nomor tiket Kakak {tid} dan perkiraan waktu tunggu sekitar {wait} menit. Rekan kami sudah memegang konteks lengkapnya, jadi Kakak tidak perlu mengulang cerita.",
        f"Tentu, Kak. Saya sudah menghubungkan Kakak dengan agen manusia (tiket {tid}). Seseorang akan membantu Kakak dalam sekitar {wait} menit."])
    return [U(noisy(q)), A(tool_call("escalate", summary=f"Pelanggan minta agen manusia: {summary_topic}."),
                           f"Pelanggan secara jelas meminta agen manusia soal {summary_topic}. Jangan berdebat; eskalasi dengan ringkasan singkat."),
            U(tool_result({"ticket_id": tid, "wait_minutes": wait})),
            A(ans, f"Tiket {tid} dibuat, waktu tunggu sekitar {wait} menit. Sampaikan dengan empati dan pastikan pelanggan tidak perlu mengulang cerita.")]

FAQ = [("kebijakan pengembalian", ["Bagaimana kebijakan pengembalian barang?", "Berapa lama batas waktu retur?", "Barang yang sudah dibeli bisa dikembalikan?"],
        "Barang bisa dikembalikan maksimal 30 hari setelah diterima. Cukup kirim nomor pesanannya dan saya akan bukakan returnya: kami kirim label retur prabayar ke email, dan dana dikembalikan ke metode pembayaran awal dalam 5-7 hari kerja."),
       ("waktu pengiriman", ["Pengiriman berapa lama?", "Kalau pesan hari ini, kapan sampai?", "Ada pengiriman ekspres?"],
        "Pengiriman reguler memakan waktu 3-5 hari kerja dan gratis ongkir untuk belanja di atas Rp500.000. Pengiriman ekspres 1-2 hari kerja."),
       ("metode pembayaran", ["Metode pembayaran apa saja yang diterima?", "Bisa bayar pakai GoPay?", "Bisa pakai QRIS?"],
        "Kami menerima transfer bank, kartu kredit, GoPay, OVO, dan QRIS."),
       ("waktu refund", ["Kapan dana refund saya masuk?", "Refund butuh berapa lama?"],
        "Setelah barang retur kami terima, dana dikembalikan ke metode pembayaran awal dalam 5-7 hari kerja. Kalau Kakak kirim nomor pesanannya, saya bisa cek retur tertentu."),
       ("gratis ongkir", ["Ada gratis ongkir?", "Ongkirnya gratis nggak?"],
        "Ada, Kak! Pengiriman reguler gratis ongkir untuk belanja di atas Rp500.000. Di bawah itu, ongkir dihitung saat checkout.")]

def conv_faq():
    topic, qs, ans = random.choice(FAQ)
    return [U(noisy(random.choice(qs))),
            A(ans, f"Ini pertanyaan umum tentang {topic}. Jawabannya ada di kebijakan toko, jadi tidak perlu tool.")]

def conv_out_of_scope():
    q = random.choice(["Bisa bantu kerjakan esai sejarah saya?", "Bagaimana cuaca besok di Bali?", "Siapa yang bakal menang pemilu berikutnya?",
                       "Kasih resep rendang dong.", "Saham apa yang sebaiknya saya beli?", "Bisa bantu PR matematika saya?"])
    a = random.choice([f"Mohon maaf, Kak, saya hanya bisa membantu soal pesanan, pengembalian, akun, dan kebijakan toko {COMPANY}. Ada yang bisa saya bantu terkait itu?",
                       f"Wah, itu di luar yang bisa saya bantu di sini. Saya asisten layanan pelanggan {COMPANY}, jadi saya bisa bantu soal pesanan, retur, pengiriman, atau akun Kakak. Ada yang mau saya cek?"])
    return [U(noisy(q)), A(a, f"Permintaan ini di luar layanan {COMPANY}. Tolak dengan sopan dan tawarkan bantuan yang berkaitan dengan toko.")]

def conv_smalltalk():
    q, a, why = random.choice([
        ("Halo", f"Halo, Kak! Saya Ava dari layanan pelanggan {COMPANY}. Ada yang bisa saya bantu hari ini?", "Pelanggan menyapa. Balas ramah dan tawarkan bantuan."),
        ("hai", f"Hai, Kak! Saya Ava, asisten layanan pelanggan {COMPANY}. Ada yang bisa saya bantu?", "Pelanggan menyapa. Balas ramah dan tawarkan bantuan."),
        ("Kamu bot ya?", f"Saya Ava, asisten AI layanan pelanggan {COMPANY}. Saya bisa cek pesanan, membuka retur, dan reset kata sandi, dan Kakak bisa minta dihubungkan ke agen manusia kapan saja.",
         "Pelanggan bertanya apakah saya bot. Jawab jujur bahwa saya asisten AI dan jelaskan apa yang bisa saya bantu.")])
    return [U(q), A(a, why)]

GENERATORS = [(conv_order_status, 20), (conv_order_no_id, 10), (conv_refund, 15), (conv_damaged, 10),
              (conv_password, 10), (conv_escalate, 10), (conv_faq, 12), (conv_out_of_scope, 8), (conv_smalltalk, 5)]

def sample_conversation():
    fns, w = zip(*GENERATORS)
    fn = random.choices(fns, weights=w)[0]
    msgs = fn()
    if fn is not conv_smalltalk and random.random() < 0.25:   # closing turn
        msgs += [U(random.choice(["makasih!", "oke, terima kasih", "sip, makasih ya"])),
                 A(random.choice(["Sama-sama, Kak! Ada lagi yang bisa saya bantu?", "Senang bisa membantu! Semoga harinya menyenangkan, Kak."]),
                   "Pelanggan berterima kasih dan masalahnya sudah selesai. Tutup dengan ramah.")]
    return {"intent": fn.__name__.replace("conv_", ""), "messages": msgs}

print(f"Generator ready: {len(GENERATORS)} intents, {len(TOOLS)} tools, seed {SEED}")

Generator ready: 9 intents, 4 tools, seed 7


## 3. Generate, validate and save
Conversations are split 90/10 **by conversation** before anything else. Every conversation is validated (roles alternate, starts with the customer, ends with Ava, every Ava turn has reasoning, tool calls are valid JSON with a known tool and stand alone, tool results are valid JSON). The files are then read back to prove they're complete.

In [ ]:
import os, json, random, datetime
from collections import Counter

conversations = [sample_conversation() for _ in range(N_CONVERSATIONS)]
for i, c in enumerate(conversations):
    c["id"] = f"conv-{i:05d}"
random.shuffle(conversations)
n_test = round(TEST_FRACTION * len(conversations))
splits = {"train": conversations[n_test:], "test": conversations[:n_test]}   # split by CONVERSATION: no leakage
for name, convs in splits.items():
    for c in convs:
        c["split"] = name

# ---------- validate every conversation before writing ----------
VALID_TOOLS = {t["name"] for t in TOOLS}
TOOL_RE = re.compile(r"<tool_call>(.*?)</tool_call>", re.S)
PREFIX = "TOOL_RESULT: "

def validate(c):
    m = c["messages"]
    assert m and m[0]["role"] == "user" and m[-1]["role"] == "assistant", "must start with user, end with assistant"
    assert all(a["role"] != b["role"] for a, b in zip(m, m[1:])), "roles must alternate"
    for x in m:
        assert x["content"].strip(), "empty content"
        if x["role"] == "assistant":
            assert x.get("reasoning", "").strip(), "assistant turn without reasoning"
            t = TOOL_RE.search(x["content"])
            if t:
                call = json.loads(t.group(1))
                assert call["name"] in VALID_TOOLS, f"unknown tool {call['name']}"
                assert x["content"].strip() == t.group(0), "a tool-call turn must contain only the tool call"
        elif x["content"].startswith(PREFIX):
            json.loads(x["content"][len(PREFIX):])

for c in conversations:
    try:
        validate(c)
    except Exception as e:
        raise ValueError(f"{c['id']} ({c['intent']}) is invalid: {e}")
print(f"✅ {len(conversations)} conversations validated")

# ---------- write JSONL + metadata ----------
def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

for name, convs in splits.items():
    write_jsonl(f"{OUT_DIR}/{name}.jsonl", convs)

def n_assistant(convs): return sum(1 for c in convs for m in c["messages"] if m["role"] == "assistant")

metadata = {
    "dataset": DATASET_NAME, "version": DATASET_VERSION, "language": "id",
    "created_at": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "seed": SEED, "generator": "Notebook 1 · synthetic templates, no real customer data",
    "system_prompt": SYSTEM_PROMPT,
    "reasoning_tag": REASONING_TAG, "reasoning_instruction": REASONING_INSTRUCTION,
    "tool_call_tag": "tool_call", "tool_result_prefix": PREFIX, "tools": TOOLS,
    "files": {name: f"{name}.jsonl" for name in splits},
    "counts": {f"{name}_conversations": len(c) for name, c in splits.items()}
              | {f"{name}_assistant_turns": n_assistant(c) for name, c in splits.items()},
    "intents": {name: dict(sorted(Counter(c["intent"] for c in convs).items())) for name, convs in splits.items()},
    "record_schema": {
        "id": "conversation id, e.g. conv-00042",
        "intent": "generator that produced the conversation",
        "split": "train | test",
        "messages": "list of {role: user|assistant, content: str}; assistant turns also have reasoning: str",
    },
}
with open(f"{OUT_DIR}/metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# ---------- read back to prove the files are complete ----------
for name, convs in splits.items():
    back = [json.loads(l) for l in open(f"{OUT_DIR}/{name}.jsonl", encoding="utf-8")]
    assert back == convs, f"{name}.jsonl does not round-trip"
print(f"✅ Saved to {OUT_DIR}:")
for fname in ["train.jsonl", "test.jsonl", "metadata.json"]:
    print(f"   {fname:14s} {os.path.getsize(f'{OUT_DIR}/{fname}') / 1024:8.1f} KB")
print(json.dumps(metadata["counts"], indent=2))

✅ 1500 conversations validated
✅ Saved to /content/drive/MyDrive/gemma4-support-agent/data:
   train.jsonl      1290.2 KB
   test.jsonl        141.1 KB
   metadata.json       3.2 KB
{
  "train_conversations": 1350,
  "test_conversations": 150,
  "train_assistant_turns": 3066,
  "test_assistant_turns": 341
}


## 4. Inspect

In [ ]:
from collections import Counter
print(f"{'intent':16s} {'train':>6s} {'test':>6s}")
for intent in sorted(metadata["intents"]["train"]):
    print(f"{intent:16s} {metadata['intents']['train'].get(intent, 0):6d} {metadata['intents']['test'].get(intent, 0):6d}")

lengths = Counter(len(c["messages"]) for c in conversations)
print("\nMessages per conversation:", dict(sorted(lengths.items())))
calls = Counter(json.loads(TOOL_RE.search(m["content"]).group(1))["name"]
                for c in conversations for m in c["messages"] if m["role"] == "assistant" and TOOL_RE.search(m["content"]))
print("Tool calls:", dict(calls))

example = next(c for c in conversations if c["intent"] == "damaged")
print(f"\n----- {example['id']} · {example['intent']} · {example['split']} -----")
for m in example["messages"]:
    if m["role"] == "assistant":
        print(f"   💭 {m['reasoning']}")
        print(f"🤖 {m['content']}")
    else:
        print(f"👤 {m['content']}")

intent            train   test
damaged             138     18
escalate            137     14
faq                 154     17
order_no_id         120     17
order_status        282     29
out_of_scope        106     13
password            139     20
refund              219     18
smalltalk            55      4

Messages per conversation: {2: 270, 4: 632, 6: 519, 8: 79}
Tool calls: {'escalate': 220, 'reset_password': 159, 'lookup_order': 448, 'start_refund': 324}

----- conv-01169 · damaged · test -----
👤 Earbud nirkabel saya datang dalam keadaan rusak. Pesanan NM-849125.
   💭 Barang tiba rusak. Ada dua jalan: refund (start_refund) atau penggantian (lewat tim manusia). Tanyakan dulu pilihan pelanggan.
🤖 Mohon maaf sekali earbud nirkabel Kakak tiba dalam keadaan rusak. Itu bukan pengalaman yang kami harapkan. Kakak lebih memilih penggantian barang atau refund penuh?
👤 Tolong kirim yang baru.
   💭 Pelanggan memilih penggantian. Tidak ada tool untuk mengirim barang pengganti, jadi saya eskal

## ✅ Next: Notebook 2
Open **Notebook 2 · Fine-tune & serve** on a **TPU** runtime. Its cell A1 has `DATA_DIR = "/content/drive/MyDrive/gemma4-support-agent/data"` — the folder written above.

> 🧪 **Extend the dataset:** add an intent (e.g. `change_address` with a new tool): describe the tool in `SYSTEM_PROMPT` and `TOOLS`, write a `conv_change_address()` that returns `U(...)` / `A(content, reasoning)` messages, register it in `GENERATORS`, bump `DATASET_VERSION`, and re-run. In Notebook 2, implement the tool in `TOOLS` (cell B2). To use real tickets instead, write them in the same record format — anonymize first.